In [56]:
!pip install pytrends #install required programs
!pip install requests

In [17]:
import pandas as pd #importing pandas and time
import time
from pytrends.request import TrendReq #imports the specific API used

pytrends = TrendReq(hl='en-US', tz=360) #connects API to Google Trends

In [22]:
kw_list = ['Tom Ford Beauty', 'Chanel Beauty', 'La Prairie', 'Dior Beauty'] #list of search terms
time.sleep(1) #creates a pause for the API to avoid error 429
pytrends.build_payload(kw_list, timeframe='2022-01-01 2025-12-31') #initial request from API using the list of search terms and set timeframe of 2022-2025
time.sleep(2) #another pause for the API to avoid the error
df = pytrends.interest_over_time() #creates a dataframe showing the search interest overtime
df #display dataframe

,Tom Ford Beauty,Chanel Beauty,La Prairie,Dior Beauty,isPartial
date,,,,,
2021-12-26,4,14,63,20,False
2022-01-02,3,16,58,20,False
2022-01-09,4,14,58,18,False
2022-01-16,3,13,53,15,False
2022-01-23,3,13,63,17,False
...,...,...,...,...,...
2025-11-30,7,29,66,53,False
2025-12-07,5,31,64,46,False
2025-12-14,5,29,73,44,False


In [23]:
df = df.drop ('isPartial', axis=1) #drops variable that will not be needed

In [57]:
labor = pd.read_csv('/Users/baileycarpenter/Downloads/file.csv') #importing labor statistics on wages and salaries


In [42]:
df.index = pd.to_datetime(df.index) #since time is the index in this dataframe, i needed to create variables for the year and quarter since that is how it is given from BLS

df["year"] = df.index.year #creates year variable
df["quarter"] = "Q" + df.index.quarter.astype(str) #separates the weeks and months into quarters based on the parameters from BLS

In [43]:
df #testing to see if code ran properly

,Tom Ford Beauty,Chanel Beauty,La Prairie,Dior Beauty,year,quarter
date,,,,,,
2021-12-26,4,14,63,20,2021,Q4
2022-01-02,3,16,58,20,2022,Q1
2022-01-09,4,14,58,18,2022,Q1
2022-01-16,3,13,53,15,2022,Q1
2022-01-23,3,13,63,17,2022,Q1
...,...,...,...,...,...,...
2025-11-30,7,29,66,53,2025,Q4
2025-12-07,5,31,64,46,2025,Q4
2025-12-14,5,29,73,44,2025,Q4


In [44]:
labor #labor statistics showing percent change in wages across quarters from 2022-2025

,Series ID,Year,Period,Label,Value
0,CIU1020000000000A,2022,Q01,2022 Qtr1,4.7
1,CIU1020000000000A,2022,Q02,2022 Qtr2,5.3
2,CIU1020000000000A,2022,Q03,2022 Qtr3,5.1
3,CIU1020000000000A,2022,Q04,2022 Qtr4,5.1
4,CIU1020000000000A,2023,Q01,2023 Qtr1,5.0
5,CIU1020000000000A,2023,Q02,2023 Qtr2,4.6
6,CIU1020000000000A,2023,Q03,2023 Qtr3,4.6
7,CIU1020000000000A,2023,Q04,2023 Qtr4,4.3
8,CIU1020000000000A,2024,Q01,2024 Qtr1,4.4
9,CIU1020000000000A,2024,Q02,2024 Qtr2,4.2


In [45]:
labor = labor.drop ('Series ID', axis=1) #removes unnecessary variable

In [47]:
labor['quarter'] = labor['Period'].str.replace('Q0', 'Q') #attempt to match the variables for data merging, shown below did not work
labor['year'] = labor['Year']

In [48]:
merged = pd.merge(df, labor, on=['year', 'quarter'], how="inner") #merging the data 

In [49]:
merged #prints the new dataframe

,Tom Ford Beauty,Chanel Beauty,La Prairie,Dior Beauty,year,quarter,Year,Period,Label,Value
0,3,16,58,20,2022,Q1,2022,Q01,2022 Qtr1,4.7
1,4,14,58,18,2022,Q1,2022,Q01,2022 Qtr1,4.7
2,3,13,53,15,2022,Q1,2022,Q01,2022 Qtr1,4.7
3,3,13,63,17,2022,Q1,2022,Q01,2022 Qtr1,4.7
4,3,14,59,18,2022,Q1,2022,Q01,2022 Qtr1,4.7
...,...,...,...,...,...,...,...,...,...,...
204,7,29,66,53,2025,Q4,2025,Q04,2025 Qtr4,3.3
205,5,31,64,46,2025,Q4,2025,Q04,2025 Qtr4,3.3
206,5,29,73,44,2025,Q4,2025,Q04,2025 Qtr4,3.3
207,4,27,68,47,2025,Q4,2025,Q04,2025 Qtr4,3.3


In [51]:
merged = merged.drop ('year', axis=1) #dropping the redundant variables
merged = merged.drop ('quarter', axis=1)
merged = merged.drop ('Label', axis=1) #label is unneded as it provides the same information as the variables year and period

In [52]:
merged #cleaned dataframe

,Tom Ford Beauty,Chanel Beauty,La Prairie,Dior Beauty,Year,Period,Value
0,3,16,58,20,2022,Q01,4.7
1,4,14,58,18,2022,Q01,4.7
2,3,13,53,15,2022,Q01,4.7
3,3,13,63,17,2022,Q01,4.7
4,3,14,59,18,2022,Q01,4.7
...,...,...,...,...,...,...,...
204,7,29,66,53,2025,Q04,3.3
205,5,31,64,46,2025,Q04,3.3
206,5,29,73,44,2025,Q04,3.3
207,4,27,68,47,2025,Q04,3.3


In [54]:
merged['luxury index'] = merged[['Tom Ford Beauty', 'Chanel Beauty', 'La Prairie', 'Dior Beauty']].mean(axis = 1) #creating an index for the average interest in luxury cosmetics

In [55]:
merged #final dataframe
merged.to_csv("", index=False)

,Tom Ford Beauty,Chanel Beauty,La Prairie,Dior Beauty,Year,Period,Value,luxury index
0,3,16,58,20,2022,Q01,4.7,24.25
1,4,14,58,18,2022,Q01,4.7,23.50
2,3,13,53,15,2022,Q01,4.7,21.00
3,3,13,63,17,2022,Q01,4.7,24.00
4,3,14,59,18,2022,Q01,4.7,23.50
...,...,...,...,...,...,...,...,...
204,7,29,66,53,2025,Q04,3.3,38.75
205,5,31,64,46,2025,Q04,3.3,36.50
206,5,29,73,44,2025,Q04,3.3,37.75
207,4,27,68,47,2025,Q04,3.3,36.50


So, I had to change my project as during Assignment 2 I did not have enough data to create charts of any kind, or really do any actual analysis with. This data was pulled using pytrends, an unofficial but well documented and supported API for Google Trends data (as the google trends API is currently in Beta testing only). The data on labor statistics was pulled from BLS, specifically the Wages and Salaries of Civillian Workers, presented in a 12-month percent change in the growth rate of wages each quarter. My new topic is examining whether depreciating wages impacts consumer spending, specifically looking at the luxury cosmetics industry and the idea of the "Lipstick Effect." The new variable created is "luxury index," which is the collective mean of the 4 beauty brands search interest to give a single measure for the demand of luxury cosmetics.

Is this the data for the new project?

Can you add more keywords for the google trends?  At least around 15 companies - then you can do your own index and that count as a novel data source via api.

The added variable is not particularly creative, but if you can make an index based on companies that might be more interesting. 

add checks to make sure the merge is correct.

Code is concise. 
grade: 92